# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imhs14/flyrank-ml-internship-starter-main/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print(con.sql(f"SELECT COUNT(*) FROM {MARCH}").fetchone()[0], "rows in March 2026 partition")


Paste your Hugging Face READ token (hf_...): ··········
9841378 rows in March 2026 partition


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
fields_contract = {
    "feature": ["gsc_impressions", "gsc_clicks", "gsc_avg_position", "content_created_at"],
    "label_or_proxy": ["(none — clustering produces its own cluster id, not a real column)"],
    "context": ["client_hash_id", "content_hash_id", "report_date"],
    "excluded": ["ga4_data_available flag rows before ga4_data_start — not enough history yet"],
}
for bucket, cols in fields_contract.items():
    print(f"{bucket}: {cols}")

feature: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'content_created_at']
label_or_proxy: ['(none — clustering produces its own cluster id, not a real column)']
context: ['client_hash_id', 'content_hash_id', 'report_date']
excluded: ['ga4_data_available flag rows before ga4_data_start — not enough history yet']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM {MARCH}
    GROUP BY 1,2,3
    HAVING c > 1
    LIMIT 5
""").df()
print(f"Duplicate grain rows found: {len(dupes)}")  # should be 0

span = con.sql(f"SELECT COUNT(*) AS n_rows, MIN(report_date) AS start, MAX(report_date) AS end FROM {MARCH}").df()
span

avail = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {MARCH}
""").df()
avail

features = con.sql(f"""
    SELECT
        f.content_hash_id,
        AVG(f.gsc_impressions)                              AS avg_impressions,
        AVG(f.gsc_avg_position)                              AS avg_position,
        SUM(f.gsc_clicks) * 1.0 / NULLIF(SUM(f.gsc_impressions), 0) AS ctr,
        COUNT(*) FILTER (WHERE f.gsc_impressions > 0)        AS days_with_impressions,
        DATE_DIFF('day', ANY_VALUE(d.content_created_date), DATE '2026-03-01') AS content_age_days
    FROM {MARCH} f
    LEFT JOIN {DIM_CONTENT} d USING (content_hash_id)
    GROUP BY f.content_hash_id
    HAVING avg_impressions IS NOT NULL
""").df()

print(f"One row = one content item, aggregated over March 2026. Shape: {features.shape}")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

One row = one content item, aggregated over March 2026. Shape: (331437, 6)


,content_hash_id,avg_impressions,avg_position,ctr,days_with_impressions,content_age_days
0,content_7a105f548d9c6916,210.419355,7.209549,0.001073,31,366
1,content_a3ea9792f793ec72,14.612903,2.987198,0.000000,31,366
2,content_36c36abc7650d7af,181.612903,6.724039,0.001066,31,366
3,content_a7da352b73b02668,159.483871,7.244844,0.002629,31,366
4,content_f39be42b42a4e8f6,1.354839,14.432540,0.000000,21,366


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.